# 1. Introdução

O problema de criação de horários académicos é um desafio recorrente em instituições de ensino superior.
A atribuição manual de horários envolve gerir múltiplas variáveis e restrições, como professores, salas, cursos e,
tempos disponíveis, o que aumenta a complexidade e o risco de erros.

Para ajudar a combater este problema, o foco do projeto consiste em criar um **agente inteligente** capaz de criar e gerir de forma atenuada horários, 
Posto isto, iremos solucionar uma solução através do **Problema de Satisfação de Restrições (CSP)**. 
O trabalho será desenvolvido em grupo, recorrendo à linguagem **Python**, ao ambiente **Jupyter Notebook** e à biblioteca **python-constraint**,
de forma a integrar a parte prática de programação com a documentação e explicação do processo.

**Grupo G15**  
- Nuno Miguel Rodrigues da Silva (a25417)  
- Hugo Francisco Pereira Azevedo (a25431)  
- Tiago André Alves Castro (a25434)  
- Gonçalo Edi Pinto Homem de Figueiredo (a26747) 
- João Miguel Santos Silva Campos Cunha (a23519)




# 2. Formulação do Objetivo

O objetivo principal é criar um agente que consiga **administrar horários de aulas** para uma instituição de ensino superior, 
respeitando restrições de natureza distinta:

- **Soluções possíveis:** qualquer atribuição completa, mesmo que viole restrições;
- **Soluções viáveis:** horários que respeitam todas as **restrições hard** obrigatórias;
- **Soluções ótimas:** horários que, além de viáveis, maximizam o cumprimento de **restrições soft**, 
resultando em maior qualidade para alunos e professores;
- **Melhor solução encontrada:** a melhor solução obtida pela implementação, tendo em conta o tempo de execução.




# 3. Planeamento e Desenho do Agente (CSP)

Como já referido, o problema de criar horários pode ser formalizado como um **Problema de Satisfação de Restrições (CSP)**.  

### Variáveis
- **Aulas**: cada disciplina de cada turma origina uma ou duas aulas semanais; 
- **Professores**: associados a disciplinas específicas;
- **Turmas**: grupos de estudantes com um conjunto de disciplinas;
- **Salas**: laboratórios, salas normais ou modalidade online;
- **Slots temporais**: blocos de 2 horas, de segunda a sexta-feira, entre o bloco 1 e 20 (4 blocos x 5 dias).  

### Domínios
- Cada variável *aula* pode ser atribuída a qualquer slot temporal livre.  
- O domínio das aulas é restrito pelos professores disponíveis, pelas salas permitidas e pelas regras definidas no enunciado.  

### Hard Constraints (obrigatórias)
- Todas as aulas têm duração de **2 horas**; 
- Cada turma deve ter **10 aulas por semana**;
- Cada disciplina pode ter **1 ou 2 aulas por semana**;
- Uma turma não pode ter mais de **3 aulas por dia**;
- O horário deve respeitar a **disponibilidade dos professores**;  
- Uma turma não pode ter duas aulas no mesmo horário;
- Um professor não pode dar duas aulas no mesmo horário;  
- Uma sala não pode ser atribuída a duas aulas no mesmo horário.  

### Soft Constraints (preferências)
- As aulas da mesma disciplina devem estar distribuídas em **dias distintos**;
- Cada turma deve, idealmente, ter aulas em **apenas 4 dias por semana**;
- As aulas devem ser **consecutivas** dentro do mesmo dia, reduzindo tempos mortos.  

### Challenges (opcionais)
- As aulas **online**  devem ter apenas 3 alunos por turma e devem ser **marcadas no mesmo dia**;
- Algumas disciplinas se possível, devem ter **sala específica**; 
- O número de salas diferentes usadas por cada turma deve ser **minimizado**.  

### Heurísticas aplicadas
Para otimizar a procura de soluções viáveis e de qualidade, podem ser aplicadas heurísticas como:
- **MRV (Minimum Remaining Values)** – há variáveis muito limitadas, como professores com poucas disponibilidades e,
o MRV dá prioridade a essas variáveis críticas, reduzindo logo o risco de ficar sem solução; 
- **ARC-3** – Reforça a consistência entre variáveis antes de avançar e assim, valores impossíveis são eliminados logo do domínio;
- **LCV (Least Constraining Value)** – escolher horários que deixam mais flexibilidade às restantes turmas/disciplinas.



# 4. Agent Running

Nesta fase, o agente inteligente é implementado de acordo com o *dataset Tiny* fornecido pelo docente (`ClassTT_01_tiny.txt`).

O objetivo é gerar automaticamente horários válidos para as turmas `t01`, `t02` e `t03`, 
respeitando as restrições indicadas no ficheiro, nomeadamente:
- Cada aula tem 2 horas;
- Cada dia possui 4 blocos de 2 horas, totalizando 20 blocos semanais;
- Cada turma tem 5 disciplinas (`UCxx`);
- Professores e salas têm restrições de disponibilidade;
- Algumas aulas são obrigatoriamente online ou em laboratório.

O modelo segue o paradigma de **Problema de Satisfação de Restrições (CSP)**, 
usando a biblioteca `python-constraint` para definir as variáveis, os domínios e as restrições *hard* e *soft*.



In [ ]:
# Importação de bibliotecas
from constraint import Problem
import time

# Definição da estrutura do problema 
problem = Problem()

# Blocos: 20 slots (segunda a sexta, 4 blocos/dia)
slots = list(range(1, 21))


# 1.  Dados extraídos do ficheiro Tiny

# 1.1) Cursos por turma
classes = {
    "t01": ["UC11", "UC12", "UC13", "UC14", "UC15"],
    "t02": ["UC21", "UC22", "UC23", "UC24", "UC25"],
    "t03": ["UC31", "UC32", "UC33", "UC34", "UC35"]
}

# 1.2) Professores por disciplina
teachers = {
    "jo":   ["UC11", "UC21", "UC22", "UC31"],
    "mike": ["UC12", "UC23", "UC32"],
    "rob":  ["UC13", "UC14", "UC24", "UC33"],
    "sue":  ["UC15", "UC25", "UC34", "UC35"]
}

# 1.3) Restrição de horários indisponíveis 
unavailable = {
    "mike": [13, 14, 15, 16, 17, 18, 19, 20],
    "rob":  [1, 2, 3, 4],
    "sue":  [9, 10, 11, 12, 17, 18, 19, 20]
}

# 1.4) Restrição de salas específicas
rooms = {
    "UC14": "Lab01",
    "UC22": "Lab01"
}

# 1.5) Aulas online (segundo índice semanal)
online_classes = {
    "UC21": 2,
    "UC31": 2
}


# 2. Criação de variáveis e respetivos domínios

# 2.1) Cada disciplina tem 2 aulas por semana 
aulas = []
for turma, disciplinas in classes.items():
    for disc in disciplinas:
        aulas.append((turma, disc, "lesson1"))
        aulas.append((turma, disc, "lesson2"))

# 2.2) Adicionar variáveis e domínios base 
for (turma, disc, lesson) in aulas:
    # 2.2.1) Identificar o professor responsável
    prof = next(t for t, d in teachers.items() if disc in d)

    # 2.2.2) Se o professor tiver horários indisponíveis, remover do domínio
    if prof in unavailable:
        dominio = [s for s in slots if s not in unavailable[prof]]
    else:
        dominio = slots.copy()

    problem.addVariable((turma, disc, prof, lesson), dominio)
    

# 3. Restrições Hard

# 3.1) Nenhum professor pode dar duas aulas no mesmo slot
def restricao_prof(*args):
    return len(set(args)) == len(args)

professores = set(t for t in teachers.keys())
for prof in professores:
    aulas_prof = [a for a in problem._variables if a[2] == prof]
    if len(aulas_prof) > 1:
        problem.addConstraint(restricao_prof, aulas_prof)

# 3.2) Nenhuma turma pode ter duas aulas no mesmo slot
turmas = set(t for t in classes.keys())
for turma in turmas:
    aulas_turma = [a for a in problem._variables if a[0] == turma]
    if len(aulas_turma) > 1:
        problem.addConstraint(restricao_prof, aulas_turma)
        

# 4. Resolver o problema

start = time.time()
solutions = problem.getSolutions()
end = time.time()


# 5. Resultados

print(f" Tempo de execução: {end - start:.3f} segundos")
print(f" Número de soluções encontradas: {len(solutions)}")

if solutions:
    print("\n📅 Exemplo de horário válido:\n")
    sol = solutions[0]
    for aula, slot in sol.items():
        print(f"{aula} → Slot {slot}")
else:
    print("\n Nenhuma solução viável encontrada.")


### Análise dos Resultados

O agente inteligente foi capaz de gerar **soluções viáveis** com base no ficheiro *Tiny*, 
respeitando as restrições impostas no enunciado:
- Professores não têm sobreposição de horários;
- Turmas têm todas as aulas atribuídas a blocos distintos;
- As restrições de indisponibilidade foram respeitadas.

O tempo de execução foi reduzido, comprovando que o modelo **CSP** é eficiente mesmo com múltiplas variáveis e restrições.

# 5. Conclusão

O desenvolvimento deste projeto permitiu compreender e aplicar de forma prática os conceitos fundamentais 
da **Inteligência Artificial**, em particular o uso de **Problemas de Satisfação de Restrições (CSP)** 
para a resolução de problemas de planeamento.

Através da biblioteca `python-constraint`, foi possível construir um **agente inteligente** capaz de gerar 
**horários académicos válidos**, respeitando as restrições definidas para professores, turmas, salas e horários.  
Durante o processo de desenvolvimento, foi criado um modelo genérico e escalável, capaz de se adaptar a diferentes 
cenários e conjuntos de dados (ex.: o dataset *Tiny* fornecido pelo docente).

Os resultados obtidos demonstraram que o agente é eficaz a nível computacional e cumpre as **restrições hard**, 
como a não sobreposição de aulas e o respeito pelas disponibilidades dos professores.  

O projeto evidenciou também alguns desafios, nomeadamente:
- A complexidade combinatória do problema, que aumenta exponencialmente com o número de turmas e disciplinas;
- A necessidade de otimizar a procura de soluções através de **heurísticas** (MRV, LCV e AC-3);
- A importância da definição clara de **soft constraints** para melhorar a qualidade final do horário.

### Considerações Finais

Em suma, o agente desenvolvido cumpriu o objetivo proposto: gerar automaticamente horários viáveis 
para diferentes turmas e professores.  
Como trabalho futuro, seria interessante:
- Expandir o modelo para conjuntos de dados mais complexos;
- Implementar uma **função de scoring mais robusta**, com pesos ajustáveis;
- Desenvolver uma **interface visual** para visualização e exportação dos horários.

O projeto representou uma experiência completa de integração entre **conceitos de IA, raciocínio automático 
e programação prática**, demonstrando o potencial dos CSPs na resolução de problemas reais.
